# Notebook 03: Translation Comparison

**Goal:** Compare 4 translation engines on the same OCR-extracted text and pick the best one.

**Engines evaluated:**
| # | Engine | Type | Key Strengths |
|---|--------|------|---------------|
| 1 | **Claude API** | Cloud LLM | Best contextual quality, handles educational tone |
| 2 | **Google Translate** | Cloud API | Fast, cheap, reliable |
| 3 | **Meta NLLB** | Local model (No Language Left Behind) | 200 languages, strong Indic support, free |
| 4 | **AI4Bharat IndicTrans2** | Local model | State-of-the-art for Indian languages |

**Input:** OCR results from `data/ocr_results/best/` (output of Notebook 02)  
**Output:** Translations saved as JSON in `data/translations/<engine>/`

In [ ]:
# Install dependencies (run once)
# !pip install anthropic
# For Google Translate: pip install google-cloud-translate
# For NLLB: pip install transformers torch sentencepiece
# For IndicTrans2: pip install transformers torch sentencepiece

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, OCR_RESULTS_DIR, TRANSLATIONS_DIR,
    load_json, save_json,
)

# Load OCR results from best engine (output of Notebook 02)
PAGE_INDEX = 0
ocr_blocks = load_json(OCR_RESULTS_DIR / "best" / f"page_{PAGE_INDEX}.json")

print(f"Loaded {len(ocr_blocks)} text blocks from OCR")
print(f"\nSample blocks:")
for b in ocr_blocks[:5]:
    print(f"  [{b['id']:3d}] {b['text'][:70]}{'...' if len(b['text']) > 70 else ''}")

In [ ]:
# ── CONFIGURE TARGET LANGUAGE ──────────────────────────────────────────────────
TARGET_LANGUAGE = "hindi"  # Options: hindi, tamil, bengali, punjabi, gujarati
# ──────────────────────────────────────────────────────────────────────────────

# Language code mappings for different APIs
LANGUAGE_CODES = {
    "hindi":    {"google": "hi", "nllb": "hin_Deva", "indictrans": "hin_Deva", "name": "Hindi"},
    "tamil":    {"google": "ta", "nllb": "tam_Taml", "indictrans": "tam_Taml", "name": "Tamil"},
    "bengali":  {"google": "bn", "nllb": "ben_Beng", "indictrans": "ben_Beng", "name": "Bengali"},
    "punjabi":  {"google": "pa", "nllb": "pan_Guru", "indictrans": "pan_Guru", "name": "Punjabi"},
    "gujarati": {"google": "gu", "nllb": "guj_Gujr", "indictrans": "guj_Gujr", "name": "Gujarati"},
}

lang_config = LANGUAGE_CODES[TARGET_LANGUAGE]
print(f"Target language: {lang_config['name']} ({TARGET_LANGUAGE})")

In [ ]:
def save_translation_results(engine_name: str, translations: list[dict], page_idx: int = 0):
    """Save translation results for an engine."""
    engine_dir = TRANSLATIONS_DIR / engine_name
    engine_dir.mkdir(parents=True, exist_ok=True)
    save_json(translations, engine_dir / f"page_{page_idx}_{TARGET_LANGUAGE}.json")
    print(f"  [{engine_name}] Saved {len(translations)} translations")


def print_translation_comparison(engine_name: str, translations: list[dict], elapsed: float):
    """Print sample translations for review."""
    print(f"\n{'='*70}")
    print(f"  {engine_name.upper()} → {TARGET_LANGUAGE.upper()} ({elapsed:.2f}s)")
    print(f"{'='*70}")
    for t in translations[:5]:
        orig = t["original"][:50] + ("..." if len(t["original"]) > 50 else "")
        trans = t["translated"][:50] + ("..." if len(t["translated"]) > 50 else "")
        print(f"  EN: {orig}")
        print(f"  {TARGET_LANGUAGE.upper()}: {trans}")
        print()

---
## Option 1: Claude API

Uses Claude as a translator with batched context. All text blocks from the page are sent in a single request for coherent, contextual translation. Best quality for educational content.

In [ ]:
import os
import anthropic


def translate_with_claude(
    text_blocks: list[dict],
    target_lang: str,
    api_key: str | None = None,
) -> list[dict]:
    """
    Translate text blocks using Claude API in a single batched request.
    
    Sends all blocks as JSON for page-level context coherence.
    """
    client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))
    
    lang_name = LANGUAGE_CODES[target_lang]["name"]
    
    # Build the input as a JSON array
    input_blocks = [
        {"id": b["id"], "text": b["text"]}
        for b in text_blocks
        if b["text"].strip()
    ]
    
    system_prompt = f"""You are a professional translator specializing in educational textbook content. 
Translate the following text segments from English to {lang_name}.

Rules:
- Maintain the educational tone and register
- For dialogue text, keep it natural and conversational in {lang_name}
- Preserve any numbers, proper nouns, or technical terms as appropriate
- Return ONLY a valid JSON array with the same structure: [{{"id": N, "translated": "..."}}]
- Do not add any explanation or extra text outside the JSON"""

    user_message = json.dumps(input_blocks, ensure_ascii=False)
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    
    # Parse response
    response_text = response.content[0].text.strip()
    # Handle potential markdown code block wrapping
    if response_text.startswith("```"):
        response_text = response_text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    translated = json.loads(response_text)
    
    # Map back to standard format with bbox
    id_to_block = {b["id"]: b for b in text_blocks}
    results = []
    for t in translated:
        block = id_to_block.get(t["id"])
        if block:
            results.append({
                "id": t["id"],
                "original": block["text"],
                "translated": t["translated"],
                "bbox": block["bbox"],
            })
    
    return results


# Run Claude translation
try:
    t0 = time.time()
    claude_results = translate_with_claude(ocr_blocks, TARGET_LANGUAGE)
    claude_time = time.time() - t0
    
    print_translation_comparison("claude", claude_results, claude_time)
    save_translation_results("claude", claude_results, PAGE_INDEX)
except Exception as e:
    print(f"Claude translation skipped: {e}")
    print("Set ANTHROPIC_API_KEY env var to enable.")
    claude_results = None

---
## Option 2: Google Translate API

Google Cloud Translation API. Fast and reliable. Translates each block individually.

**Setup:** `pip install google-cloud-translate` and set `GOOGLE_APPLICATION_CREDENTIALS`.

In [ ]:
def translate_with_google(
    text_blocks: list[dict],
    target_lang: str,
) -> list[dict]:
    """
    Translate text blocks using Google Cloud Translation API.
    
    Requires: pip install google-cloud-translate
    """
    from google.cloud import translate_v2 as translate
    
    client = translate.Client()
    target_code = LANGUAGE_CODES[target_lang]["google"]
    
    results = []
    for block in text_blocks:
        if not block["text"].strip():
            continue
        
        response = client.translate(block["text"], target_language=target_code, source_language="en")
        
        results.append({
            "id": block["id"],
            "original": block["text"],
            "translated": response["translatedText"],
            "bbox": block["bbox"],
        })
    
    return results


# Run Google Translate
try:
    t0 = time.time()
    google_results = translate_with_google(ocr_blocks, TARGET_LANGUAGE)
    google_time = time.time() - t0
    
    print_translation_comparison("google_translate", google_results, google_time)
    save_translation_results("google_translate", google_results, PAGE_INDEX)
except Exception as e:
    print(f"Google Translate skipped: {e}")
    print("Install google-cloud-translate and set GOOGLE_APPLICATION_CREDENTIALS to enable.")
    google_results = None

---
## Option 3: Meta NLLB (No Language Left Behind)

Meta's open-source model supporting 200 languages. Strong for Indian languages. Runs locally — no API key needed, but requires GPU for good performance.

**Setup:** `pip install transformers torch sentencepiece`

In [ ]:
def translate_with_nllb(
    text_blocks: list[dict],
    target_lang: str,
    model_name: str = "facebook/nllb-200-distilled-600M",  # Use 1.3B or 3.3B for better quality
) -> list[dict]:
    """
    Translate text blocks using Meta's NLLB model locally.
    
    Models (quality vs speed tradeoff):
    - facebook/nllb-200-distilled-600M  (fast, decent quality)
    - facebook/nllb-200-1.3B            (good balance)
    - facebook/nllb-200-3.3B            (best quality, needs GPU)
    
    Requires: pip install transformers torch sentencepiece
    """
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    
    target_code = LANGUAGE_CODES[target_lang]["nllb"]
    
    print(f"  Loading NLLB model: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    print(f"  Model loaded.")
    
    results = []
    for block in text_blocks:
        if not block["text"].strip():
            continue
        
        # Tokenize with source language
        tokenizer.src_lang = "eng_Latn"
        inputs = tokenizer(block["text"], return_tensors="pt", max_length=512, truncation=True)
        
        # Generate translation
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=512,
        )
        translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
        
        results.append({
            "id": block["id"],
            "original": block["text"],
            "translated": translated_text,
            "bbox": block["bbox"],
        })
    
    return results


# Run NLLB
try:
    t0 = time.time()
    nllb_results = translate_with_nllb(ocr_blocks, TARGET_LANGUAGE)
    nllb_time = time.time() - t0
    
    print_translation_comparison("nllb", nllb_results, nllb_time)
    save_translation_results("nllb", nllb_results, PAGE_INDEX)
except Exception as e:
    print(f"NLLB skipped: {e}")
    print("Install transformers, torch, sentencepiece to enable.")
    nllb_results = None

---
## Option 4: AI4Bharat IndicTrans2

State-of-the-art model specifically designed for Indian language translation by AI4Bharat/IIT Madras. Uses the same HuggingFace transformers interface.

**Setup:** `pip install transformers torch sentencepiece`

In [ ]:
def translate_with_indictrans2(
    text_blocks: list[dict],
    target_lang: str,
    model_name: str = "ai4bharat/indictrans2-en-indic-dist-200M",  # Distilled, faster
) -> list[dict]:
    """
    Translate text blocks using AI4Bharat's IndicTrans2 model.
    
    Models:
    - ai4bharat/indictrans2-en-indic-dist-200M  (distilled, fast)
    - ai4bharat/indictrans2-en-indic-1B          (full, best quality)
    
    Requires: pip install transformers torch sentencepiece
    """
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    
    target_code = LANGUAGE_CODES[target_lang]["indictrans"]
    
    print(f"  Loading IndicTrans2 model: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
    print(f"  Model loaded.")
    
    results = []
    for block in text_blocks:
        if not block["text"].strip():
            continue
        
        # IndicTrans2 uses a specific input format
        # The model expects: "Translate from English to Hindi: <text>"
        # But with the HF pipeline, we set src/tgt language codes
        tokenizer.src_lang = "eng_Latn"
        inputs = tokenizer(block["text"], return_tensors="pt", max_length=512, truncation=True)
        
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=512,
        )
        translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
        
        results.append({
            "id": block["id"],
            "original": block["text"],
            "translated": translated_text,
            "bbox": block["bbox"],
        })
    
    return results


# Run IndicTrans2
try:
    t0 = time.time()
    indictrans_results = translate_with_indictrans2(ocr_blocks, TARGET_LANGUAGE)
    indictrans_time = time.time() - t0
    
    print_translation_comparison("indictrans2", indictrans_results, indictrans_time)
    save_translation_results("indictrans2", indictrans_results, PAGE_INDEX)
except Exception as e:
    print(f"IndicTrans2 skipped: {e}")
    print("Install transformers, torch, sentencepiece to enable.")
    indictrans_results = None

---
## Side-by-Side Comparison

Compare translations from all available engines in a table format.

In [ ]:
# Collect available results
all_translations = {}
all_trans_times = {}

for name, results, t in [
    ("Claude", claude_results, claude_time if claude_results else 0),
    ("Google Translate", google_results, google_time if google_results else 0),
    ("NLLB", nllb_results, nllb_time if nllb_results else 0),
    ("IndicTrans2", indictrans_results, indictrans_time if indictrans_results else 0),
]:
    if results:
        all_translations[name] = {t["id"]: t for t in results}
        all_trans_times[name] = t

# Display comparison table for first N blocks
N_DISPLAY = 8
print(f"\n{'='*100}")
print(f"TRANSLATION COMPARISON — First {N_DISPLAY} blocks → {TARGET_LANGUAGE.upper()}")
print(f"{'='*100}")

for block in ocr_blocks[:N_DISPLAY]:
    bid = block["id"]
    original = block["text"][:80]
    print(f"\n[Block {bid}] ORIGINAL: {original}")
    print("-" * 80)
    for engine_name, trans_dict in all_translations.items():
        if bid in trans_dict:
            translated = trans_dict[bid]["translated"][:80]
            print(f"  {engine_name:<18}: {translated}")
    print()

# Summary
print(f"\n{'Engine':<18} {'Blocks':>7} {'Time (s)':>10}")
print("-" * 40)
for name in all_translations:
    n = len(all_translations[name])
    t = all_trans_times[name]
    print(f"{name:<18} {n:>7} {t:>10.2f}")

## Select Best Translation Engine

In [ ]:
# Auto-detect which translation engines produced results
ENGINE_DIRS = {
    "claude": "claude",
    "google_translate": "google_translate",
    "nllb": "nllb",
    "indictrans2": "indictrans2",
}

available_engines = {}
for name, dirname in ENGINE_DIRS.items():
    engine_dir = TRANSLATIONS_DIR / dirname
    if engine_dir.exists() and any(engine_dir.iterdir()):
        available_engines[name] = dirname

if not available_engines:
    raise RuntimeError("No translation engine produced results! Run at least one engine cell above.")

print(f"Available engines: {list(available_engines.keys())}")

# ── SELECT YOUR BEST ENGINE (set to None for auto-select, or pick one) ────
BEST_TRANSLATOR = None  # e.g., "claude", "google_translate", "nllb", "indictrans2"
# ──────────────────────────────────────────────────────────────────────────────

if BEST_TRANSLATOR is None:
    BEST_TRANSLATOR = list(available_engines.keys())[0]
    print(f"Auto-selected: {BEST_TRANSLATOR}")

if BEST_TRANSLATOR not in available_engines:
    raise ValueError(f"'{BEST_TRANSLATOR}' has no results. Choose from: {list(available_engines.keys())}")

import shutil

source_dir = TRANSLATIONS_DIR / available_engines[BEST_TRANSLATOR]
best_dir = TRANSLATIONS_DIR / "best"

if best_dir.exists():
    shutil.rmtree(best_dir)
best_dir.mkdir(parents=True, exist_ok=True)

for f in source_dir.iterdir():
    shutil.copy2(f, best_dir / f.name)

print(f"Copied {BEST_TRANSLATOR} results to data/translations/best/")
print(f"\n✓ Translation selection complete. Next notebook: 04_inpainting_comparison.ipynb")